# Project 6 — Model Monitoring & Data Drift

Objective: Models degrade silently as real-world data shifts away from what they were trained on. This project simulates that shift and uses Evidently AI to detect it automatically.

Approach:
1. Load the cleaned credit risk data (same cleaning logic as `pipeline.py` from Project 1).
2. Treat a held-out slice as the reference dataset (what the model was trained/validated on).
3. Create a current dataset — a copy of it with `person_income` artificially multiplied by 1.5, simulating a scenario where incoming applicants suddenly report much higher incomes (e.g. a data pipeline bug, or a genuine economic shift).
4. Run Evidently's Data Drift report comparing the two, and save it as an HTML report.

Output: `data_drift_report.html` — Artifact 6 on the checklist.

In [3]:
import sys
!{sys.executable} -m pip install evidently

  Using cached cryptography-49.0.0-cp311-abi3-win_amd64.whl.metadata (4.3 kB)
   ---------------------------------------- 0.0/11.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.7 MB ? eta -:--:--
   ---------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mlflow 3.14.0 requires cryptography<49,>=43.0.0, but you have cryptography 49.0.0 which is incompatible.
pyopenssl 24.2.1 requires cryptography<44,>=41.0.5, but you have cryptography 49.0.0 which is incompatible.
s3fs 2024.6.1 requires fsspec==2024.6.1.*, but you have fsspec 2026.6.0 which is incompatible.
streamlit 1.38.0 requires protobuf<6,>=3.20, but you have protobuf 6.33.6 which is incompatible.
streamlit 1.38.0 requires rich<14,>=10.14.0, but you have rich 15.0.0 which is incompatible.


In [4]:
import pandas as pd
from evidently import Report, Dataset, DataDefinition
from evidently.presets import DataDriftPreset

## 1. Load and clean the data

Same cleaning rule as Project 1 (`pipeline.py`): drop physically impossible values using fixed thresholds, not statistics learned from the data.

In [5]:
df = pd.read_csv("C:\\Users\\jesut\\Downloads\\archive (5)\\credit_risk_dataset.csv")

before = len(df)
df = df[df["person_age"] <= 100]
df = df[(df["person_emp_length"].isna()) | (df["person_emp_length"] <= 60)]
df = df.reset_index(drop=True)

print(f"Loaded {before} rows, dropped {before - len(df)} invalid rows, {len(df)} remain.")

Loaded 32581 rows, dropped 7 invalid rows, 32574 remain.


## 2. Create the reference and current datasets

- **Reference:** a random 50% slice of the cleaned data — stands in for "what the model saw during training/validation."
- **Current:** the other 50% slice, but with `person_income` multiplied by 1.5 — stands in for "new incoming production data" where something has shifted.

Using two genuinely separate slices (not the same rows twice) keeps this realistic — in production you're always comparing genuinely different batches of data over time.

In [6]:
shuffled = df.sample(frac=1.0, random_state=42).reset_index(drop=True)
midpoint = len(shuffled) // 2

reference = shuffled.iloc[:midpoint].reset_index(drop=True)
current = shuffled.iloc[midpoint:].reset_index(drop=True).copy()

# Simulate data drift: incoming applicants suddenly reporting 50% higher income
# (e.g. could be a real economic shift, or a unit conversion bug in an upstream data pipeline --
# both are realistic reasons a monitoring system like this would need to catch)
current["person_income"] = current["person_income"] * 1.5

print(f"Reference set: {reference.shape}")
print(f"Current set:   {current.shape}")
print()
print("Reference person_income mean:", round(reference["person_income"].mean(), 2))
print("Current person_income mean:  ", round(current["person_income"].mean(), 2))

Reference set: (16287, 12)
Current set:   (16287, 12)

Reference person_income mean: 65857.66
Current person_income mean:   98848.95


## 3. Run the drift report

Evidently statistically compares each column's distribution in `current` against `reference` (using tests like Kolmogorov-Smirnov for numeric columns, chi-square for categorical) and flags which ones have drifted beyond a normal threshold.

In [7]:
reference_dataset = Dataset.from_pandas(reference, data_definition=DataDefinition())
current_dataset = Dataset.from_pandas(current, data_definition=DataDefinition())

report = Report([DataDriftPreset()])
result = report.run(current_dataset, reference_dataset)

result.save_html("data_drift_report.html")
print("Report saved to data_drift_report.html")

Report saved to data_drift_report.html


## 4. Quick summary in the notebook

The full interactive report is in the HTML file, but here's a quick sanity check that `person_income` was correctly flagged as drifted while unrelated columns were not.

In [8]:
result_dict = result.dict()

for metric in result_dict["metrics"]:
    name = metric["metric_name"]
    if "ValueDrift" in name:
        print(f"{name} -> {metric['value']}")
    elif "DriftedColumnsCount" in name:
        print(f"{name} -> {metric['value']}")

DriftedColumnsCount(drift_share=0.5) -> {'count': 1.0, 'share': 0.08333333333333333}
ValueDrift(column=person_age,method=Wasserstein distance (normed),threshold=0.1) -> 0.010201718266610487
ValueDrift(column=person_income,method=Wasserstein distance (normed),threshold=0.1) -> 0.6149285873215206
ValueDrift(column=person_emp_length,method=Wasserstein distance (normed),threshold=0.1) -> 0.017791854061493462
ValueDrift(column=loan_amnt,method=Wasserstein distance (normed),threshold=0.1) -> 0.007762406356809268
ValueDrift(column=loan_int_rate,method=Wasserstein distance (normed),threshold=0.1) -> 0.009930741859382204
ValueDrift(column=loan_percent_income,method=Wasserstein distance (normed),threshold=0.1) -> 0.010069018424083247
ValueDrift(column=cb_person_cred_hist_length,method=Wasserstein distance (normed),threshold=0.1) -> 0.00811633791517626
ValueDrift(column=person_home_ownership,method=Jensen-Shannon distance,threshold=0.1) -> 0.007102454085405101
ValueDrift(column=loan_intent,method

**How to read this:** for each column, a small p-value (well below 0.05) means the distribution has changed significantly — that's drift. `person_income` should show a very small p-value (drift detected), while unrelated columns like `person_age` should show a p-value close to 1.0 (no drift, as expected, since we didn't touch them).

Open `data_drift_report.html` in your browser for the full interactive report, with distribution plots for every column — that's what goes in your portfolio/README as Artifact 6.